# INNOBEST Huánuco 2026 — Entrenamiento 100 épocas / 150 fotos / GPU T4
**No entrenar sin imágenes auténticas y licenciadas.** Los modelos y métricas se guardan en Google Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# DESCOMPRIMIR el ZIP si aún no existe la carpeta (el ZIP debe estar en Mi unidad).
from pathlib import Path
import zipfile
base = Path("/content/drive/MyDrive")
archivo_zip = base / "innobest_huanuco_2026_base.zip"
carpeta = base / "innobest_huanuco_2026"
if not carpeta.exists():
    if not archivo_zip.exists():
        raise FileNotFoundError(f"Primero sube {archivo_zip} a Drive")
    with zipfile.ZipFile(archivo_zip) as zf:
        zf.extractall(base)
print("Proyecto:", carpeta)
for nombre in ["ara_militaris", "pauxi_koepckeae", "spizaetus_isidori"]:
    (carpeta / "dataset" / nombre).mkdir(parents=True, exist_ok=True)
print("Coloca 50 fotos originales en cada subcarpeta de dataset y continúa.")


In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Activa T4 en Entorno de ejecución → Cambiar tipo de entorno de ejecución.")
!nvidia-smi


In [ ]:
# Se ejecutarán 100 épocas (80 de cabecera y 20 de ajuste fino).
# El mejor modelo se selecciona por validación; no hay parada temprana.
!pip -q install -U "scikit-learn>=1.4,<2" "pandas>=2,<3" "Pillow>=10,<13"
%cd /content/drive/MyDrive/innobest_huanuco_2026
!python entrenar.py --datos dataset --salida modelos --epocas-cabecera 80 --epocas-ajuste 20 --batch 16


In [ ]:
# Revisar métricas reales; NO aparecen valores inventados.
import json
archivo = carpeta / "modelos" / "metricas.json"
if archivo.exists():
    m = json.loads(archivo.read_text(encoding="utf-8"))
    print("Exactitud prueba:", m["accuracy_test"])
    print("F1 macro:", m["f1_macro_test"])
    print("GPU:", m["gpu"])
    print("Épocas ejecutadas:", m["epocas_ejecutadas"], "/", m["epocas_solicitadas"])
    print("Mejor época:", m["mejor_epoca_global"])
    print("Fotos de prueba:", m["n_prueba"])
else:
    print("Entrena el modelo antes de consultar métricas.")
